In [ ]:
from pathlib import Path
import sys
import os
import datetime
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import cmocean.cm as cmo

In [ ]:
ds = xr.open_dataset("data/delayed_SB2121_M2.nc")
df = ds.to_pandas()

### Resample

Applying a 60 second median filter to align timestamps and smooth out sensor noise

In [ ]:
df = df.resample('60s').median()

### Identify deployment time
We detect the deployment and recovery of the sailbuoy by when conductivity passes 36

In [ ]:
mid_time = df.index.mean()
out_of_water = df.index[df.CNDC<36]
in_water_time = max(out_of_water[out_of_water<mid_time])
in_water_time
out_water_time = min(out_of_water[out_of_water>mid_time])
out_water_time
df = df[np.logical_and(df.index>in_water_time + np.timedelta64(1,'h'), df.index<out_water_time - np.timedelta64(1,'h'))]

In [ ]:
fix, ax = plt.subplots()
ax.plot(df.index, df.CNDC)
ax.axvline(in_water_time,c='C1')
ax.axvline(out_water_time,c='C1')
ax.set(ylabel='Conductivity')

In [ ]:
ds['TEMP'].attrs

In [ ]:
fix, ax = plt.subplots(3,1, figsize=(10,9))
var_names = ['TEMP', 'WIND_SPEED', 'PRESSURE_AIR']
for i , var_name in enumerate(var_names):
    attrs = ds[var_name].attrs
    label = f"{var_name} [{attrs['units']}]"
    ax[i].plot(df.index, df[var_name], label=label)
    ax[i].legend()
ax[0].set(title="SB2121 met data");